# C5 — Calibração e Escolha de Limiar

Calibra os finalistas e escolhe o limiar operacional usando **somente
treino e validação**. Se `HONEST_CALIB_SPLIT = True`, a validação é
dividida em `val_cal` (ajusta calibrador) e `val_meas` (mede Brier,
escolhe limiar). Não abre o teste e não escreve no YAML.

**Pré-requisito:** `pip install -e .` com scikit-learn ≥ 1.6.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact, split_partitions
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas, brier
from src.evaluation.calibration import curva_confiabilidade, ece, mce, brier_decomposicao
from src.models.factory import build_finalista
from src.models.calibration import calibrar, escolher_limiar

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

SEED      = int(C.SEED)
G7_STATUS = str(C.cfg("gates", "G7_calibracao", "status") or "pendente")
G8_STATUS = str(C.cfg("gates", "G8_limiar", "status") or "pendente")

print(C.resumo())

## Carregar artefacto e preparar validação

Se `HONEST_CALIB_SPLIT = True`, `val_cal` ajusta o calibrador e
`val_meas` mede — evita o viés otimista de calibrar e medir no mesmo
conjunto. Caso contrário, ambos são a validação inteira.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(ROOT_DIR, expected_kind=COHORT_KIND)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES].copy()
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

check_leakage(X_train)
check_leakage(X_val)

HONEST_CALIB_SPLIT = True
CALIB_SPLIT_FRAC   = 0.5
CALIB_SPLIT_SEED   = SEED

if HONEST_CALIB_SPLIT:
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - CALIB_SPLIT_FRAC,
        random_state=CALIB_SPLIT_SEED,
        stratify=y_val.to_numpy(),
    )
    X_val_cal  = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal  = y_val.iloc[idx_cal].reset_index(drop=True)
    X_val_meas = X_val.iloc[idx_meas].reset_index(drop=True)
    y_val_meas = y_val.iloc[idx_meas].reset_index(drop=True)
    CALIB_SPLIT_HASH = hashlib.sha256(
        np.asarray(idx_cal, dtype=np.int64).tobytes()
    ).hexdigest()[:16]
else:
    X_val_cal, y_val_cal = X_val, y_val
    X_val_meas, y_val_meas = X_val, y_val
    CALIB_SPLIT_HASH = None

n_teste = int((df[PARTICAO].astype(str) == "teste").sum())

display(pd.DataFrame({
    "bloco":      ["treino", "val_cal", "val_meas"],
    "n":          [len(X_train), len(X_val_cal), len(X_val_meas)],
    "positivos":  [int(y_train.sum()), int(y_val_cal.sum()), int(y_val_meas.sum())],
}))

print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {n_teste:,} (fechado)")
if CALIB_SPLIT_HASH:
    print(f"split_hash  : {CALIB_SPLIT_HASH}")

## Calibrar finalistas e escolher limiar

Para cada finalista da `selecao_modelos.finalistas`, treina no
treino, calibra cada método candidato em `val_cal`, mede Brier em
`val_meas` e escolhe o melhor. Depois escolhe o limiar no mesmo
`val_meas`.

O YAML usa `hist_gradient_boosting`; a fábrica usa `boosting` —
mapeamento abaixo, sem duplicar lógica de construção.

In [ ]:
ALIASES_FINALISTA = {
    "logistic":               "logistic",
    "boosting":               "boosting",
    "hist_gradient_boosting": "boosting",
}

selecao_cfg = C.cfg("selecao_modelos") or {}
FINALISTAS  = list(selecao_cfg.get("finalistas") or [])
if not FINALISTAS:
    raise RuntimeError("selecao_modelos.finalistas está vazio no YAML.")

desconhecidos = [n for n in FINALISTAS if n not in ALIASES_FINALISTA]
if desconhecidos:
    raise ValueError(f"Finalistas desconhecidos: {desconhecidos}")

calibracao_cfg      = C.cfg("calibracao") or {}
CALIBRACAO_ATIVA    = bool(calibracao_cfg.get("ativa", True))
METODOS_CANDIDATOS  = list(calibracao_cfg.get("metodos_candidatos") or ["sigmoid", "isotonic"])

limiar_cfg            = C.cfg("limiar") or {}
LIMIAR_METRICA        = limiar_cfg.get("metrica") or "f1"
LIMIAR_MIN_PRECISION  = limiar_cfg.get("min_precision")

print(f"finalistas          : {FINALISTAS}")
print(f"calibracao ativa    : {CALIBRACAO_ATIVA} ({METODOS_CANDIDATOS})")
print(f"limiar metrica      : {LIMIAR_METRICA} (min_precision={LIMIAR_MIN_PRECISION})")

resultados = {}

for nome_yaml in FINALISTAS:
    pipe = build_finalista(ALIASES_FINALISTA[nome_yaml])
    pipe.fit(X_train, y_train)
    prob_base = pipe.predict_proba(X_val_meas)[:, 1]
    brier_base = float(brier(y_val_meas, prob_base))

    calibracoes = {}
    if CALIBRACAO_ATIVA:
        for metodo in METODOS_CANDIDATOS:
            try:
                cal = calibrar(pipe, X_val_cal, y_val_cal, method=metodo)
                prob = cal.predict_proba(X_val_meas)[:, 1]
                calibracoes[metodo] = {
                    "pipeline":  cal,
                    "prob_meas": prob,
                    "brier":     float(brier(y_val_meas, prob)),
                }
            except Exception as exc:
                print(f"  {nome_yaml}/{metodo}: falhou ({exc!r})")

    if calibracoes:
        metodo = min(calibracoes, key=lambda m: calibracoes[m]["brier"])
        escolha = {"metodo": metodo, **calibracoes[metodo]}
    else:
        escolha = {"metodo": None, "pipeline": pipe,
                   "prob_meas": prob_base, "brier": brier_base}

    resultados[nome_yaml] = {
        "brier_base":    brier_base,
        "prob_base_meas": prob_base,
        "calibracoes":   calibracoes,
        "escolha":       escolha,
    }

    res = escolher_limiar(
        y_val_meas,
        escolha["prob_meas"],
        metrica=LIMIAR_METRICA,
        min_precision=LIMIAR_MIN_PRECISION,
    )
    resultados[nome_yaml]["limiar"] = res

tabela = pd.DataFrame([
    {"finalista": n,
     "metodo_calibracao": r["escolha"]["metodo"],
     "brier_base": r["brier_base"],
     "brier_calibrado": r["escolha"]["brier"],
     "ganho_brier": r["brier_base"] - r["escolha"]["brier"],
     "limiar": r["limiar"]["limiar"],
     "metrica": r["limiar"]["metrica"],
     "valor_metrica": r["limiar"]["valor"]}
    for n, r in resultados.items()
])
display(tabela)

## Diagnósticos e persistência

Curva de confiabilidade, decomposição de Brier, ECE e MCE em
`val_meas` com o método escolhido. Persistimos tudo em `outputs/`.

In [ ]:
diagnosticos = {}
linhas_diag  = []

for nome, r in resultados.items():
    prob = r["escolha"]["prob_meas"]
    d = {
        "curva":  curva_confiabilidade(y_val_meas, prob, n_bins=10),
        "decomp": brier_decomposicao(y_val_meas, prob, n_bins=10),
        "ece":    float(ece(y_val_meas, prob, n_bins=10)),
        "mce":    float(mce(y_val_meas, prob, n_bins=10)),
    }
    diagnosticos[nome] = d
    linhas_diag.append({
        "finalista":   nome,
        "metodo":      r["escolha"]["metodo"] or "sem_calibracao",
        "ece":         d["ece"],
        "mce":         d["mce"],
        "brier":       d["decomp"]["brier"],
        "reliability": d["decomp"]["reliability"],
        "resolution":  d["decomp"]["resolution"],
    })

display(pd.DataFrame(linhas_diag))

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"C5_calibracao_limiar_{manifest['cohort_kind']}_{timestamp}"

caminho_tabela       = TABLES_DIR      / f"{prefixo}_escolhas.csv"
caminho_predicoes    = PREDICTIONS_DIR / f"{prefixo}_predicoes_val_meas.parquet"
caminho_manifesto    = TABLES_DIR      / f"{prefixo}_manifesto.json"

tabela.to_csv(caminho_tabela, index=False, encoding="utf-8-sig")

artefatos_diag = {}
for nome, d in diagnosticos.items():
    c_curva = TABLES_DIR / f"{prefixo}_{nome}_curva.csv"
    d["curva"].to_csv(c_curva, index=False, encoding="utf-8-sig")
    resultados[nome]["limiar"]["tabela"].to_csv(
        TABLES_DIR / f"{prefixo}_{nome}_limiar.csv", index=False, encoding="utf-8-sig",
    )
    artefatos_diag[nome] = {
        "curva":         str(c_curva),
        "ece":           d["ece"],
        "mce":           d["mce"],
        "brier_decomp":  {k: float(v) for k, v in d["decomp"].items()},
    }

predicoes = []
for nome, r in resultados.items():
    predicoes.append(pd.DataFrame({
        "y_true": y_val_meas.to_numpy(), "y_prob": r["prob_base_meas"],
        "modelo": f"{nome}_base", "cohort_kind": manifest["cohort_kind"],
    }))
    if r["escolha"]["metodo"]:
        predicoes.append(pd.DataFrame({
            "y_true": y_val_meas.to_numpy(), "y_prob": r["escolha"]["prob_meas"],
            "modelo": f"{nome}_{r['escolha']['metodo']}", "cohort_kind": manifest["cohort_kind"],
        }))

pred_val = pd.concat(predicoes, ignore_index=True)
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c5 = {
    "cohort_kind":        manifest["cohort_kind"],
    "dataset_sha256":     manifest["dataset_sha256"],
    "feature_columns":    FEATURES,
    "target_column":      ALVO,
    "partition_column":   PARTICAO,
    "n_treino":           int(len(treino)),
    "n_validacao":        int(len(validacao)),
    "n_val_cal":          int(len(X_val_cal)),
    "n_val_meas":         int(len(X_val_meas)),
    "n_teste_fechado":    n_teste,
    "honest_calib_split": bool(HONEST_CALIB_SPLIT),
    "calib_split_frac":   float(CALIB_SPLIT_FRAC),
    "calib_split_seed":   int(CALIB_SPLIT_SEED),
    "calib_split_hash":   CALIB_SPLIT_HASH,
    "finalistas":         FINALISTAS,
    "calibracao_ativa":   CALIBRACAO_ATIVA,
    "metodos_candidatos": METODOS_CANDIDATOS,
    "escolhas": {
        n: {"metodo": r["escolha"]["metodo"],
            "brier_base":      r["brier_base"],
            "brier_calibrado": r["escolha"]["brier"],
            "limiar":          r["limiar"]["limiar"],
            "limiar_metrica":  r["limiar"]["metrica"]}
        for n, r in resultados.items()
    },
    "diagnosticos":       artefatos_diag,
    "g7_status":          G7_STATUS,
    "g8_status":          G8_STATUS,
    "teste_aberto":       False,
    "yaml_atualizado":    False,
    "artefatos": {
        "tabela":              str(caminho_tabela),
        "predicoes_val_meas":  str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c5, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print(f"Tabela     : {caminho_tabela}")
print(f"Predições  : {caminho_predicoes}")
print(f"Manifesto  : {caminho_manifesto}")

In [ ]:
melhor_nome      = max(resultados, key=lambda n: resultados[n]["limiar"]["auc"])
metodo_escolhido = resultados[melhor_nome]["escolha"]["metodo"]
limiar_escolhido = resultados[melhor_nome]["limiar"]

proposta_yaml = "\n".join([
    "# Proposta de atualização do YAML — C5 (não aplicada automaticamente).",
    f"# cohort_kind: {manifest['cohort_kind']}",
    f"# G7/G8 no momento: {G7_STATUS}/{G8_STATUS}",
    f"# finalista sugerido (maior AUC em val_meas): {melhor_nome}",
    "calibracao:",
    f"  metodo: {metodo_escolhido}",
    "limiar:",
    f"  otimo: {limiar_escolhido['limiar']}",
    f"  metrica: {limiar_escolhido['metrica']}",
])
print(proposta_yaml)

with Run(
    modelo="C5_calibracao_limiar",
    parametros={
        "cohort_kind":        manifest["cohort_kind"],
        "finalistas":         FINALISTAS,
        "calibracao_ativa":   CALIBRACAO_ATIVA,
        "metodos_candidatos": METODOS_CANDIDATOS,
        "honest_calib_split": HONEST_CALIB_SPLIT,
        "limiar_metrica":     LIMIAR_METRICA,
        "min_precision":      LIMIAR_MIN_PRECISION,
        "teste_aberto":       False,
        "yaml_atualizado":    False,
    },
    dataset={
        "n_treino":        int(len(treino)),
        "n_val_cal":       int(len(X_val_cal)),
        "n_val_meas":      int(len(X_val_meas)),
        "n_teste_fechado": n_teste,
    },
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C5 — calibração e limiar escolhidos na validação. Teste não consultado.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**{
        f"{n}_{sufixo}": float(valor)
        for n, res in resultados.items()
        for sufixo, valor in [
            ("brier_base", res["brier_base"]),
            ("brier_calibrado", res["escolha"]["brier"]),
            ("limiar", res["limiar"]["limiar"]),
            ("auc", res["limiar"]["auc"]),
        ]
    })

print("Run registrado em:", r.pasta)